# Pump Power Scan Notebook
## This notebook can be used to look at the pump probe signal for different powers. Each run that is loaded into this notebook should only consist of one time point, post-time zero.

## Imports

In [ ]:
import xrayscatteringtools as xrst
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
xrst.enable_underscore_cleanup()

## Loading runs

In [ ]:
###############################################
runNumbers = [] # <- this must be a list of int. For later plots to work properly, there should be at least 2 runs here
folders = xrst.get_data_paths(runNumbers) # Defaults to the info in config.yaml. You can overwrite this with strings, character arrays, or lists of either.
pump_energies = xrst.get_config_for_runs(runNumbers,'pump_energy','energy') # Defaults to the info in config.yaml. You can overwrite this.
###############################################
# (1) keys_to_combine: some keys loaded for each shot & stored per shot 
# (2) keys_to_sum: some keys loaded per each run and added 
# (3) keys_to_check : check if some keys exist and have same values in all runs and load these keys 
_keys_to_combine = [
    # Azimuthal Average
    'jungfrau4M/azav_azav',
    # Photon energy
    'ebeam/photon_energy',
    # Upstream Diode
    'ipm_dg2/sum',
    # Gas detectors
    'gas_detector/f_11_ENRC', 'gas_detector/f_22_ENRC',
    # Light status
    'lightStatus/laser', 'lightStatus/xray',
]
_keys_to_sum = [
]
_keys_to_check = [
    'UserDataCfg/jungfrau4M/cmask', # Combined mask
    'UserDataCfg/jungfrau4M/azav__azav_q', # q bin centers
    'UserDataCfg/jungfrau4M/azav__azav_qbin', # q bin size
    'UserDataCfg/jungfrau4M/azav__azav_qbins', # q bin edges
    # These keys are typically not needed, but feel free to uncomment them.
    # 'UserDataCfg/jungfrau4M/azav__azav_idxq',
    # 'UserDataCfg/jungfrau4M/azav__azav_idxphi',
    # 'UserDataCfg/jungfrau4M/azav__azav_nphi',
    # 'UserDataCfg/jungfrau4M/azav__azav_matrix_q',
    # 'UserDataCfg/jungfrau4M/azav__azav_matrix_phi',
]
##### Load the data in #####
_data = xrst.combineRuns(runNumbers, folders, _keys_to_combine, _keys_to_sum, _keys_to_check, verbose=False)  # this is the function to load the data with defined keys
############################
# String for nice things
runNumbersRange = xrst.compress_ranges(runNumbers)
runType = xrst.get_config_for_runs(runNumbers[0],'samples','sample') # Default to information in the first run you load.
niceTitle = f"{xrst.get_config('expNumber')} : {'Run' if np.size(runNumbers)==1 else 'Runs'} {runNumbersRange} : {runType}"

# Jungfrau4M Data
azav = np.squeeze(_data['jungfrau4M/azav_azav'])
J4MSum = np.nansum(azav,axis=tuple(range(1, azav.ndim)))
q = _data['UserDataCfg/jungfrau4M/azav__azav_q'] # q bin centers
qbin = _data['UserDataCfg/jungfrau4M/azav__azav_qbin'] # q bin-size
qbins = _data['UserDataCfg/jungfrau4M/azav__azav_qbins'] # q bins edges

# Event codes / run indicator
laserOn = _data['lightStatus/laser'].astype(bool)  # laser on events
xrayOn = _data['lightStatus/xray'].astype(bool)  # xray on events
run_indicator = _data['run_indicator'] # run indicator for each shot

# X-ray beam diagnostics
dg2 = _data['ipm_dg2/sum']   # upstream diode x-ray intensity
pulse_energy = _data['gas_detector/f_11_ENRC']   # xray energy from gas detector (not calibrated to actual values)
photon_energy = _data['ebeam/photon_energy']    # x-ray energy energy in eV
# Print total shots
_total_shots = len(run_indicator)
print("Total shots: ", _total_shots)
print(f'Pulse energies {pump_energies}')

## 1D Filtering

In [ ]:
########## Different filter cutoffs##########
_J4M_cutoff = [0.4, 1];
_dg2_cutoff = [0.4, 1];
_pulse_energy_cutoff = [0.5, 1.5] # In mJ!!!
_plot_display = 'log' # 'linear'
#############################################

# Precomputing the normalized values
_J4MSumNorm = J4MSum/np.nanmax(J4MSum);
_dg2Norm = dg2/np.nanmax(dg2);

plt.figure(figsize=[17,10]) 

plt.subplot(1,3,1)
plt.hist(_J4MSumNorm,bins=200,range=[0,np.nanmax(_J4MSumNorm)]);
plt.axvline(_J4M_cutoff[0],color='r',linestyle='--')
plt.axvline(_J4M_cutoff[1],color='r',linestyle='--')
plt.axvspan(_J4M_cutoff[0],_J4M_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('J4M Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(1,3,2)
plt.hist(_dg2Norm,bins=200,range=[0,np.nanmax(_dg2Norm)]);
plt.axvline(_dg2_cutoff[0],color='r',linestyle='--')
plt.axvline(_dg2_cutoff[1],color='r',linestyle='--')
plt.axvspan(_dg2_cutoff[0],_dg2_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('DG2 Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(1,3,3)
plt.hist(pulse_energy,bins=200,range=[0,np.nanmax(pulse_energy)]);
plt.axvline(_pulse_energy_cutoff[0],color='r',linestyle='--')
plt.axvline(_pulse_energy_cutoff[1],color='r',linestyle='--')
plt.axvspan(_pulse_energy_cutoff[0],_pulse_energy_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('Pulse Energy Histogram')
plt.xlabel('mJ')
plt.ylabel('Counts')
plt.legend()

plt.suptitle(niceTitle)
plt.show()

goodIdx = np.logical_and.reduce([
    _J4M_cutoff[0] <= _J4MSumNorm,
    _J4MSumNorm <= _J4M_cutoff[1],
    _dg2_cutoff[0] <= _dg2Norm,
    _dg2Norm <= _dg2_cutoff[1],
    _pulse_energy_cutoff[0] <= pulse_energy,
    pulse_energy <= _pulse_energy_cutoff[1],
    xrayOn
])
# Displaying how much data was kept due to this filtering
_counts, _ = np.histogram(goodIdx.astype(int),[0,1,2]);
print(f'goodIdx data represents {_counts[1]/np.sum(_counts)*100:.2f}% of the total shots. ({_counts[1]} out of {np.sum(_counts)}).')

## 2D Filtering

In [ ]:
#####################################################################
# Define your bounds as dictionaries with slope 'm' and intercept 'b'
line_upper = {'m': 1.0, 'b': 0.15}
line_lower = {'m': 1.0, 'b': -0.15}
_bypass = False
#####################################################################

# Normalize axes to fraction of max
_dg2_subset = dg2[goodIdx]
_j4m_subset = J4MSum[goodIdx]

_dg2_min = np.nanmin(_dg2_subset)
_dg2_max = np.nanmax(_dg2_subset - _dg2_min)
_dg2_norm = (_dg2_subset - _dg2_min) / _dg2_max

_j4m_min = np.nanmin(_j4m_subset)
_j4m_max = np.nanmax(_j4m_subset - _j4m_min)
_j4m_norm = (_j4m_subset - _j4m_min) / _j4m_max

plt.figure(figsize=[12,5])

# 2D Histogram
plt.subplot(1,2,1)
plt.hist2d(_dg2_norm, _j4m_norm, bins=200, norm=LogNorm())
plt.colorbar()
plt.xlabel('DG2 (fraction of max)')
plt.ylabel('J4M (fraction of max)')
plt.title('J4M vs DG2 Histogram, Good Shots')

# Plot the arbitrary linear boundary lines
_x = np.linspace(0, 1, 500)
_y_lower_plot = line_lower['m'] * _x + line_lower['b']
_y_upper_plot = line_upper['m'] * _x + line_upper['b']

plt.plot(_x, _y_lower_plot, '--', color='r')
plt.plot(_x, _y_upper_plot, '--', color='r')

plt.xlim(0,1)
plt.ylim(0,1)

# 1D Histogram (Relative Position between lines)
plt.subplot(1,2,2)

# Wxpected Y values for every data point on both boundary lines
_y_bound_lower = line_lower['m'] * _dg2_norm + line_lower['b']
_y_bound_upper = line_upper['m'] * _dg2_norm + line_upper['b']

# Calculate relative position between the lines
# 1e-9 to prevent division by zero in case lines intersect
_relative_position = (_j4m_norm - _y_bound_lower) / (_y_bound_upper - _y_bound_lower + 1e-9)

# Plotting range is set slightly wider than [0, 1] to show the rejected data tails
plt.hist(_relative_position, bins=200, range=(-1.5, 2.5))
plt.axvline(0, color='r', linestyle='--')
plt.axvline(1, color='r', linestyle='--')
plt.axvspan(0, 1, color='r', alpha=0.2, label='"Good" Data')
plt.legend()
plt.xlabel('Relative Position (0=Lower Line, 1=Upper Line)')
plt.ylabel('Counts')
plt.title('Data Position Between Bounds')

plt.suptitle(niceTitle)
plt.tight_layout()
plt.show()

# make the boolean mask
if _bypass:
    
    _subset_pass = np.ones_like(_j4m_subset)
    print('Bypassing this filter')
else:
    _subset_pass = np.logical_and(
        _j4m_norm >= _y_bound_lower,
        _j4m_norm <= _y_bound_upper
    )
    _counts, _ = np.histogram(_subset_pass.astype(int),[0,1,2]);
    print(f'This filter retained {_counts[1]/np.sum(_counts)*100:.2f}% of the already filtered shots. ({_counts[1]} out of {np.sum(_counts)}).')
goodIdx_2 = np.zeros_like(goodIdx, dtype=bool)
goodIdx_2[goodIdx] = _subset_pass
# Displaying how much data was kept due to this filtering

## Determine normalization method

In [ ]:
##########################################################################################
_norm_method = 'dg2' # Supports 'dg2', 'none', 'q_range'
_q_range = [2.5,4] # This is only used if you select 'q-range' as the normalization method
##########################################################################################
if _norm_method == 'none':
    azav_norm = azav
elif _norm_method == 'dg2':
    with np.errstate(divide='ignore', invalid='ignore'):
        azav_norm = azav/dg2[:,np.newaxis]
elif _norm_method == 'q_range':
    # Determine valid q range idxs for filtereing the signal.
    _qidx_norm = (_q_range[0]<q) & (q<_q_range[1])
    with np.errstate(divide='ignore', invalid='ignore'):
        azav_norm = azav / np.nanmean(azav[:,_qidx_norm],axis=1)[:,np.newaxis]

## Compute percent difference for each run

In [ ]:
# Preallocation of arrays for pdiff 
_azav_laseroff_run = np.zeros(((azav[0,:].size), np.unique(run_indicator).size)) # I(q) for Laser off shots per run
_azav_laseron_run = np.zeros(((azav[0,:].size), np.unique(run_indicator).size)) # I(q) for Laser on shots per run

# Computing the avg azav off and on for each run
for _run_idx, _run in enumerate(runNumbers):
    _off_idx = (run_indicator == _run) & ~laserOn & goodIdx_2
    _on_idx = (run_indicator == _run) & laserOn & goodIdx_2
    _azav_laseroff_run[:,_run_idx] = np.nanmean(azav_norm[_off_idx],axis=0)
    _azav_laseron_run[:,_run_idx] = np.nanmean(azav_norm[_on_idx],axis=0)
print('Done')
# Compute percent difference
pdiff_run = (azav_laseron_run - azav_laseroff_run) / azav_laseroff_run

## Sort by pump energy, and plot the percent difference patterns

In [ ]:
# Get indices that sort pump energies
sorted_idx = np.argsort(pump_energies)
plt.figure()
for _run in sorted_idx:
    plt.plot(q, dI_I_run[:, _run], label=f'Run {runNumbers[_run]}: {pump_energies[_run]} uJ')
plt.axhline(0, color='grey', linestyle='--')
plt.xlabel('q')
plt.ylabel('dI/I')
plt.legend()
plt.title(niceTitle)
plt.show()

## Determine a q-range to sum over, plot and fit the |sum|

In [ ]:
###########################
_q_range = [0,4]
_fit_range = [0,10] # in uJ
###########################
plt.figure(figsize=[12,5])
plt.subplot(1,2,1)
for _run_idx in sorted_idx:
    plt.plot(q, dI_I_run[:, _run_idx], label=f'Run {runNumbers[_run_idx]}: {pump_energies[_run_idx]} uJ')
plt.axhline(0, color='grey', linestyle='--')
plt.axvline(_q_range[0],color='r',linestyle='--')
plt.axvline(_q_range[1],color='r',linestyle='--')
plt.axvspan(_q_range[0],_q_range[1],color='r',alpha=0.2)
plt.axvspan()
plt.xlabel('q')
plt.ylabel('dI/I')
plt.legend()

_q_mask = (_q_range[0] <= q) & (q <= _q_range[1])
_sum = [np.nansum(np.abs(dI_I_run[_q_mask, _run_idx])) for _run_idx in sorted_idx]
_pump_energy_mask = (_fit_range[0] <= pump_energies) & (pump_energies <= _fit_range[1])
_coefficients = np.polyfit(pump_energies[_pump_energy_mask],_sum[_pump_energy_mask], 1)
_fit_line = np.polyval(_coefficients,pump_energies)

plt.subplot(1,2,2)
plt.scatter(pump_energies, _sum, label='Data')
plt.plot(pump_energies,_fit_line,color='r',label=f'Fit, {_fit_range[0]} to {_fit_range[1] uJ}')
plt.figure()
plt.xlabel('Pump pulse energy, (uJ))
plt.ylabel('|Sum| of dI/I')
plt.title(f'Summed q range, {_q_range[0]} to {_q_range[1]}')
plt.legend()

plt.suptitle(niceTitle)
plt.show()